## Speed & Cost Analysis
Operational benchmarks across all three models.
Extrapolates 200-review sample results to a 50,000-review production workload.
Includes projected API costs for GPT-OSS-120B and the recommended Gemma 4 31B.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Set a nice theme for our charts
sns.set_theme(style="whitegrid")

In [ ]:
file_path_meta_llama = "../data/ollama_metadata_llama_final.csv"
file_path_meta_gemma = "../data/ollama_metadata_gemma4e4b_final.csv"
file_path_meta_gpt = "../data/groq_metadata_gpt.csv"

df_meta_llama = pd.read_csv(file_path_meta_llama)
df_meta_gemma = pd.read_csv(file_path_meta_gemma)
df_meta_gpt = pd.read_csv(file_path_meta_gpt)

In [ ]:
# Convert Ollama nanoseconds to seconds
df_meta_llama['duration_sec'] = df_meta_llama['total_duration_ns'] / 1e9
df_meta_gemma['duration_sec'] = df_meta_gemma['total_duration_ns'] / 1e9

# Groq is already in seconds, but let's rename it to match
df_meta_gpt['duration_sec'] = df_meta_gpt['total_time']

In [ ]:
plt.figure(figsize=(10, 4))

plot_data = pd.DataFrame({
    'Duration': pd.concat([
        df_meta_llama['duration_sec'],
        df_meta_gemma['duration_sec'],
        df_meta_gpt['duration_sec']
    ], ignore_index=True),

    'Source': (
        ['Llama (Ollama)'] * len(df_meta_llama) +
        ['Gemma (Ollama)'] * len(df_meta_gemma) +
        ['GPT (Groq)'] * len(df_meta_gpt)
    )
})

sns.boxplot(
    x='Duration',
    y='Source',
    hue='Source',
    data=plot_data,
    palette="coolwarm",
    legend=False
)

plt.title("Time per Review")
plt.xlabel("Seconds")
plt.show()

In [ ]:
backlog_size = 50000

avg_llama = df_meta_llama['duration_sec'].mean()
avg_gemma = df_meta_gemma['duration_sec'].mean()
avg_groq = df_meta_gpt['duration_sec'].mean()

llama_time_hours = (avg_llama * backlog_size) / 3600 # Convert seconds to hours
gemma_time_hours = (avg_gemma * backlog_size) / 3600 
groq_time_hours = (avg_groq * backlog_size) / 3600

print(f"--- Time to Process {backlog_size} Reviews ---")
print(f"Local (llama): {llama_time_hours:.1f} hours ({llama_time_hours/24:.1f} days)")
print(f"Local (gemma): {gemma_time_hours:.1f} hours ({gemma_time_hours/24:.1f} days)")
print(f"Cloud (Groq):   {groq_time_hours:.1f} hours")

## 3.&nbsp; The Economics (Token Costs) 💰

In [ ]:
# Pricing (per 1M tokens)
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60

# Average tokens per review
avg_input = df_meta_gpt['input_tokens'].mean()
avg_output = df_meta_gpt['output_tokens'].mean()

# Total tokens for backlog
total_input_millions = (avg_input * backlog_size) / 1_000_000
total_output_millions = (avg_output * backlog_size) / 1_000_000

# Total Cost
cost_input = total_input_millions * PRICE_INPUT
cost_output = total_output_millions * PRICE_OUTPUT
total_bill = cost_input + cost_output

print(f"--- Projected Bill for {backlog_size} Reviews ---")
print(f"Total Input Cost:  ${cost_input:.2f}")
print(f"Total Output Cost: ${cost_output:.2f}")
print(f"TOTAL ESTIMATED:   ${total_bill:.2f}")

## Gemma 4 31b 
The Recommendation

In [ ]:
# Pricing (per 1M tokens)
PRICE_INPUT = 0.13
PRICE_OUTPUT = 0.38

# Average tokens per review
avg_input = df_meta_gpt['input_tokens'].mean()
avg_output = df_meta_gpt['output_tokens'].mean()

# Total tokens for backlog
total_input_millions = (avg_input * backlog_size) / 1_000_000
total_output_millions = (avg_output * backlog_size) / 1_000_000

# Total Cost
cost_input = total_input_millions * PRICE_INPUT
cost_output = total_output_millions * PRICE_OUTPUT
total_bill = cost_input + cost_output

print(f"--- Projected Bill for {backlog_size} Reviews ---")
print(f"Total Input Cost:  ${cost_input:.2f}")
print(f"Total Output Cost: ${cost_output:.2f}")
print(f"TOTAL ESTIMATED:   ${total_bill:.2f}")